# Titanic: de una tabla a una matriz numérica

NumPy nos permitió operar con una matriz cuyas columnas eran todas temperaturas.
En Titanic hay columnas numéricas y de texto, valores faltantes e identificadores.
Usaremos pandas para trabajar con la tabla etiquetada y NumPy cuando necesitemos
una matriz de números. No entrenaremos un modelo en esta sesión.

## Contenido

1. Leer e inspeccionar el CSV.
2. Seleccionar filas y columnas; reconocer ausentes.
3. Preparar variables y resumir grupos.
4. Unir una pequeña tabla de códigos.
5. Convertir columnas numéricas a NumPy y repetir el trabajo desde `main.py`.

Abre esta notebook desde la carpeta `sesion-02-pandas`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from titanic.analysis import (
    FEATURE_COLUMNS,
    load_passengers,
    numerical_arrays,
    prepare_passengers,
    survival_by_group,
)

csv_path = Path.cwd()/"Titanic-Dataset.csv"
if not csv_path.is_file() or not (Path.cwd() / "main.py").is_file():
    raise FileNotFoundError("Start this notebook from sesion-02-pandas")
print(csv_path.name)

Titanic-Dataset.csv


## 1. Leer una tabla

`read_csv` usa los nombres de la primera fila como etiquetas de columnas. Cada
fila representa un pasajero. Un `DataFrame` puede tener columnas de distintos
tipos; un `ndarray` numérico de NumPy no conserva estas etiquetas. Empezamos por
ver unas filas, la forma y los tipos antes de transformarlo.

`passengers["Age"]` devuelve una `Series`: una columna con su índice.
`passengers[["Age"]]` pasa una lista de nombres y conserva un `DataFrame`
de una columna. El índice identifica filas en pandas y no equivale a
`PassengerId`, que por ahora es otra columna.

[Documentación de `read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) ·
[estructuras de pandas](https://pandas.pydata.org/docs/user_guide/dsintro.html).

In [2]:
mi_serie = pd.Series([1, 2, 3, 4, 5])
mi_serie.name = "mi_serie"
mi_serie

0    1
1    2
2    3
3    4
4    5
Name: mi_serie, dtype: int64

In [3]:
df = pd.DataFrame(
    {
        "Nombres": ["Alice", "Bob", "Charlie"],
        "Edades": [25, 30, 35],
        "Sexo": ["F", "M", "M"],
    }
)

#metodo head() para mostrar las primeras filas del DataFrame, siendo 5 el valor por defecto, pero se puede especificar otro número de filas a mostrar como argumento.

df.head()  # Muestra las primeras 5 filas del DataFrame
# df.head(2)  # Muestra las primeras 2 filas del DataFrame

,Nombres,Edades,Sexo
0,Alice,25,F
1,Bob,30,M
2,Charlie,35,M


In [4]:
df.tail()  # Muestra las últimas 5 filas del DataFrame

,Nombres,Edades,Sexo
0,Alice,25,F
1,Bob,30,M
2,Charlie,35,M


In [5]:
passengers = pd.read_csv(csv_path)
print(passengers.shape)
print(passengers.dtypes)
print("Series shape:", passengers["Age"].shape)
print("DataFrame shape:", passengers[["Age"]].shape)
passengers.head()

(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object
Series shape: (891,)
DataFrame shape: (891, 1)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


El módulo `titanic.analysis` usa el mismo método de lectura y además comprueba
que existan las columnas necesarias para la aplicación. Más adelante compararemos las operaciones de la notebook con las funciones
que reutiliza la aplicación.

In [6]:
loaded = load_passengers(csv_path)
assert loaded.shape == passengers.shape
print(list(loaded.columns))

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


## 2. Ausentes, duplicados y selección

`isna()` produce una tabla de booleanos; `sum()` cuenta los verdaderos de cada
columna. `PassengerId` identifica pasajeros, así que revisamos si hay IDs
repetidos. Un nombre no es una clave segura: dos personas podrían compartirlo.

In [7]:
missing_by_column = passengers.isna().sum()
print(missing_by_column[missing_by_column > 0])
print("Repeated passenger IDs:", passengers["PassengerId"].duplicated().sum())
assert int(missing_by_column["Age"]) == 177

Age         177
Cabin       687
Embarked      2
dtype: int64
Repeated passenger IDs: 0


`.loc[filas, columnas]` selecciona por etiquetas. La condición siguiente
elige pasajeros de primera clase con edad registrada. Conservamos los nombres de
las columnas y sabemos qué representa cada valor. Los paréntesis alrededor de
cada comparación permiten combinar condiciones con `&`.

[Selección con pandas](https://pandas.pydata.org/docs/user_guide/indexing.html).

In [8]:
first_class_with_age = passengers.loc[
    (passengers["Pclass"] == 1) & passengers["Age"].notna(),
    ["PassengerId", "Pclass", "Age", "Fare"],
]
print(first_class_with_age.shape)
first_class_with_age.head()

(186, 4)


,PassengerId,Pclass,Age,Fare
1,2,1,38.0,71.2833
3,4,1,35.0,53.1000
6,7,1,54.0,51.8625
11,12,1,58.0,26.5500
23,24,1,28.0,35.5000


### Ejercicio 1: describir los ausentes

Calcula cuántas edades y puertos de embarque (`Embarked`) faltan. Comprueba
que son 177 y 2, respectivamente. Muestra las filas donde falta
`Embarked`, conservando `PassengerId`, `Pclass` y `Embarked`. Explica por qué
`Cabin` no sería una buena columna para empezar un ejemplo de imputación simple.

In [9]:
#Ejercicio 1: Encuentra los pasajeros que tienen valores faltantes en la columna "Embarked" y muestra sus PassengerId, Pclass y Embarked.

# Write your solution here.
print(passengers[["Age", "Embarked"]].isna().sum())
missing_ports = passengers.loc[
    passengers["Embarked"].isna(), 
    ["PassengerId", "Pclass" ,"Embarked"],
    ]

missing_ports

Age         177
Embarked      2
dtype: int64


,PassengerId,Pclass,Embarked
61,62,1,NaN
829,830,1,NaN


In [10]:
passengers.describe() #resumen estadístico de las columnas numéricas del DataFrame, incluyendo la media, desviación estándar, valores mínimo y máximo, y percentiles.

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## 3. Preparar columnas y resumir grupos

La mediana de `Age` ofrece un valor de reemplazo sencillo para este ejercicio.
`median()` calcula la mediana omitiendo ausentes y `fillna` reemplaza esos
ausentes. Creamos una copia de las columnas que usará la aplicación.
Antes de rellenar, registramos qué filas tenían la edad ausente: así no confundimos
un valor observado con uno añadido. La nueva columna `family_size` cuenta al
pasajero, sus hermanos o cónyuge (`SibSp`) y sus padres o hijos (`Parch`). Estas
sumas se aplican a columnas completas, sin un ciclo por pasajero.

Aquí describimos y representamos datos; si más adelante se evalúa un modelo, la
imputación deberá ajustarse únicamente con los datos de entrenamiento.

[Valores ausentes en pandas](https://pandas.pydata.org/docs/user_guide/missing_data.html).

In [11]:
from titanic.analysis import REQUIRED_COLUMNS

prepared = passengers.loc[:, list(REQUIRED_COLUMNS)].copy()
prepared["age_missing"] = prepared["Age"].isna()
median_age = prepared["Age"].median()
prepared["age_filled"] = prepared["Age"].fillna(median_age)
prepared["family_size"] = prepared["SibSp"] + prepared["Parch"] + 1
print("Median age:", median_age)
print(prepared[["Age", "age_missing", "age_filled", "family_size"]].head(10))
assert int(prepared["age_missing"].sum()) == 177

Median age: 28.0
    Age  age_missing  age_filled  family_size
0  22.0        False        22.0            2
1  38.0        False        38.0            2
2  26.0        False        26.0            1
3  35.0        False        35.0            2
4  35.0        False        35.0            1
5   NaN         True        28.0            1
6  54.0        False        54.0            1
7   2.0        False         2.0            5
8  27.0        False        27.0            3
9  14.0        False        14.0            2


In [12]:
passengers.groupby("Sex").mean(numeric_only=True)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
Sex,,,,,,,
female,431.028662,0.742038,2.159236,27.915709,0.694268,0.649682,44.479818
male,454.147314,0.188908,2.389948,30.726645,0.429809,0.235702,25.523893


In [13]:
passengers.groupby("Sex")[["Survived","Age"]].mean(numeric_only=True)

,Survived,Age
Sex,,
female,0.742038,27.915709
male,0.188908,30.726645


`groupby` reúne filas por las columnas indicadas. `size` cuenta pasajeros;
`mean` sobre `Survived`, que contiene ceros y unos, da la proporción observada de
unos en cada grupo. Estas son descripciones de los datos, no efectos causales de
la clase o el sexo.

`as_index=False` mantiene las columnas de agrupación como columnas de la tabla.
En `.agg`, `passengers=("Survived", "size")` indica el nombre de salida, la
columna de entrada y la operación. `dropna=False` conserva también los grupos
con clase o sexo ausente, si aparecieran en otra entrada.

[Guía oficial de `groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html).

In [14]:
summary = prepared.groupby(["Pclass", "Sex"], as_index=False, dropna=False).agg(
    passengers=("Survived", "size"),
    survival_rate=("Survived", "mean"),
)
print(summary)
assert int(summary["passengers"].sum()) == len(prepared)

   Pclass     Sex  passengers  survival_rate
0       1  female          94       0.968085
1       1    male         122       0.368852
2       2  female          76       0.921053
3       2    male         108       0.157407
4       3  female         144       0.500000
5       3    male         347       0.135447


### Ejercicio 2: comparar grupos

Con `groupby`, produce una tabla con una fila por `Pclass` que contenga la
cantidad de pasajeros y la proporción observada de supervivencia. Comprueba que
los conteos suman 891. Identifica cuál clase tiene la proporción más alta y
redacta una frase descriptiva sin afirmar que la clase fue la causa.

In [15]:
#Ejercicio 2: Agrupa los pasajeros por la clase de pasajero (Pclass) y calcula el número total de pasajeros y el número de sobrevivientes en cada clase. Muestra el resultado en un DataFrame con las columnas "Pclass", "passengers" y "survivors".

# Write your solution here.

Summary_2 = prepared.groupby(["Pclass"], as_index=False, dropna=False).agg(
    passengers=("Survived", "size"),
    survivors=("Survived", "sum")
)
print(Summary_2.sum())
print(Summary_2)
assert int(Summary_2["passengers"].sum()) == len(prepared)

Pclass          6
passengers    891
survivors     342
dtype: int64
   Pclass  passengers  survivors
0       1         216        136
1       2         184         87
2       3         491        119


### Ejercicio 3: una nueva variable

Crea en una copia de `prepared` la columna booleana `travel_alone`, verdadera
cuando `family_size` vale 1. Cuenta cuántos pasajeros cumplen esa condición.
Comprueba que modificar la copia no añade una columna a `prepared`.

In [16]:
# Write your solution here. -- AQUI ME FALTA REVISAR PORQUE NO ME DA EL RESULTADO ESPERADO.

prepared_copy = prepared.copy()
print(prepared_copy.head(5))
prepared_copy["travel_alone"] = prepared_copy["family_size"].sum() == 1
print(prepared_copy["travel_alone"].sum())

   PassengerId  Survived  Pclass     Sex   Age  SibSp  Parch     Fare  \
0            1         0       3    male  22.0      1      0   7.2500   
1            2         1       1  female  38.0      1      0  71.2833   
2            3         1       3  female  26.0      0      0   7.9250   
3            4         1       1  female  35.0      1      0  53.1000   
4            5         0       3    male  35.0      0      0   8.0500   

  Embarked  age_missing  age_filled  family_size  
0        S        False        22.0            2  
1        C        False        38.0            2  
2        S        False        26.0            1  
3        S        False        35.0            2  
4        S        False        35.0            1  
0


## 4. Unir una tabla pequeña

El CSV guarda `Embarked` como código. Construimos una tabla de consulta y la
unimos con `merge`. `how="left"` conserva todos los pasajeros; los dos códigos
ausentes siguen sin puerto conocido. Esta unión sirve para añadir una etiqueta,
no para inventar la información faltante.

`on="Embarked"` indica la columna común. `validate="many_to_one"` comprueba
que la tabla de puertos tenga un solo registro por código, aunque muchos
pasajeros compartan ese código. Así evitamos multiplicar pasajeros al unir.

[Uniones en pandas](https://pandas.pydata.org/docs/user_guide/merging.html).

In [17]:
ports = pd.DataFrame(
    {
        "Embarked": ["C", "Q", "S"],
        "port_name": ["Cherbourg", "Queenstown", "Southampton"],
    }
)
with_ports = prepared.merge(ports, on="Embarked", how="left", validate="many_to_one")
print(with_ports[["PassengerId", "Embarked", "port_name"]].head())
print("Unmatched ports:", with_ports["port_name"].isna().sum())
assert len(with_ports) == len(prepared)

   PassengerId Embarked    port_name
0            1        S  Southampton
1            2        C    Cherbourg
2            3        S  Southampton
3            4        S  Southampton
4            5        S  Southampton
Unmatched ports: 2


## 5. De pandas a NumPy

Una tabla es útil para trabajar con nombres, texto y ausentes. Una matriz NumPy
resulta adecuada para operaciones numéricas por ejes y sirve de puente hacia los
tensores de la siguiente sesión. Seleccionamos solo cuatro columnas numéricas y
convertimos explícitamente a `float32`; las etiquetas `Survived` quedan aparte.
Al convertir, la matriz pierde los nombres de columna: por eso guardamos el orden
en `FEATURE_COLUMNS`.

[`DataFrame.to_numpy`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html).

In [18]:
features = prepared.loc[:, list(FEATURE_COLUMNS)].to_numpy(dtype=np.float32)
labels = prepared["Survived"].to_numpy(dtype=np.int64)
print(FEATURE_COLUMNS)
print("Features:", features.shape, features.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Finite features:", np.isfinite(features).all())
print("Mean by feature:", features.mean(axis=0))
assert features.shape == (891, 4)
assert labels.shape == (891,)

('Pclass', 'age_filled', 'Fare', 'family_size')
Features: (891, 4) float32
Labels: (891,) int64
Finite features: True
Mean by feature: [ 2.308642  29.361582  32.20421    1.9046016]


### Ejercicio 4: comprobar el puente

Localiza la posición de `age_filled` en `FEATURE_COLUMNS` y calcula la media de
esa columna en `features` con NumPy. Compárala con la media de
`prepared["age_filled"]` mediante `np.testing.assert_allclose` con tolerancia
absoluta `1e-4` y `rtol=0` (sin tolerancia relativa). Puedes usar
`FEATURE_COLUMNS.index("age_filled")` para localizar la columna. Explica por qué las dos cifras pueden diferir muy ligeramente.

In [19]:
#Ejercicio 4: Crea un DataFrame que contenga la edad promedio de los pasajeros agrupados por la clase de pasajero (Pclass) y el sexo (Sex). El DataFrame debe tener las columnas "Pclass", "Sex" y "average_age".
# Write your solution here.

age_position = FEATURE_COLUMNS.index("age_filled")
numpy_mean = features[:, age_position].mean()
pandas_mean = prepared["age_filled"].mean()
np.testing.assert_allclose(numpy_mean, pandas_mean, rtol=1e-5)

print(f"Mean age (NumPy): {numpy_mean}")
print(f"Mean age (Pandas): {pandas_mean}")

Mean age (NumPy): 29.361581802368164
Mean age (Pandas): 29.36158249158249


## 6. Repetir el procesamiento

Las funciones del módulo contienen estas mismas operaciones, junto con
comprobaciones de entrada. La celda siguiente verifica que sus resultados
coincidan con los que acabamos de construir.

Ejecuta `uv run --locked python main.py` en la terminal de este proyecto. El
programa usa las mismas funciones y escribe una tabla preparada, un resumen y
dos arreglos. Abre los archivos en `outputs/` para verificar sus formas y
columnas. Continúa con el [ejercicio 5](./PRACTICA.md) para ampliar el reporte.

In [20]:
from_module = prepare_passengers(passengers)
pd.testing.assert_frame_equal(prepared, from_module)
pd.testing.assert_frame_equal(summary, survival_by_group(from_module))
module_features, module_labels = numerical_arrays(from_module)
np.testing.assert_array_equal(features, module_features)
np.testing.assert_array_equal(labels, module_labels)
print("Notebook and module results match")

Notebook and module results match
